<a href="https://colab.research.google.com/github/LeandroHCarvalho/agentes-2026-2-equipe-agentes_especiais/blob/main/ApresentacaoTrabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Equipe Agentes Especiais

### Integrantes:
* Beatriz
* Lara
* Leandro



### Nosso problema:

Hoje, quem vai alugar um imóvel precisa entender rapidamente todas as obrigações, prazos e taxas do contrato, mas os documentos possuem linguagem jurídica complexa e requisitos técnicos extensos, o que causa insegurança ao assinar e surpresas financeiras indesejadas durante a locação.



---



### Quem sofre com isso?
Pessoas que precisam alugar uma moradia.



---



### Como se resolve hoje?
Hoje quem precisa alugar uma moradia, fica a mercê, pois os contratos tem linguagem juridica complexa e falta uma compreensão melhor sobre as legislações vigentes. Quem tem acesso a um advogado consegue se resguardar de alguns abusos, mas a grande maioria das pessoas, simplesmente assinam o contrato mesmo com insegurança, pois precisam de um local para morar.



---



## PEAS

| | |
|---|---|
| **P**erformance — como se mede sucesso, de forma verificável | Precisão na extração de dados, detecção de riscos e divergências, clareza na lingagem, curto tempo de resposta |
| **E**nvironment — sobre que dados, sistemas e documentos opera | Entrada de contratos, jurisprudencias, e leis vigentes sobre o tema |
| **A**ctuators — que ações o agente pode executar | Avaliar e comparar os contratos com as leis vigentes |
| **S**ensors — o que ele recebe como entrada | Contratos de alugel |



---



Analisando a complexidade do problema que nos sugerimos, optamos por utilizar o Workflow + Base Vetorial (RAG).

##Porque o RAG?

Antes de falar o porque, o que seria o RAG?

##Retrieval-Augmented Generation **ou** Geração Aumentada por Recuperação.

A parte de Generation utilizando esses fundamentos para produzir uma análise final ainda não está implementada nesse notebook.

Isso é importante: não diga que o código atual já faz a geração final baseada no RAG, porque a etapa mostrada termina no Retrieval.

##Vamos utilizar o RAG porque vamos utilizar as legislações vigentes no brasil a respeito de inquilinato e leis referentes presentes no Código Civil e também no Código de Defesa do Consumidor.

#Mãos na massa!

Vamos ao código, que é o que todo mundo quer ver! 🥰

---



# Instalação das dependências

In [81]:
# ============================================
# Instalação das dependências
# ============================================

!pip install -q openai python-dotenv pydantic tenacity pdfplumber scikit-learn numpy reportlab



---



##O que é uma dependência e o que elas fazem?

É um conjunto de códigos, funções e rotinas pré-escritas por outros desenvolvedores que você pode reutilizar no seu próprio programa.

Em vez de você escrever do zero um algoritmo complexo para calcular matemática vetorial, ler a estrutura interna de um PDF ou fazer requisições de rede para um servidor, você importa uma biblioteca que já resolveu esse problema de forma otimizada e testada


---


###Cada biblioteca em nosso projeto possui uma função:

###`openai`




Biblioteca oficial da OpenAI. Permite que seu código se conecte às APIs dos modelos de IA da empresa (como GPT-4o, DALL-E, modelos de embeddings, etc.).
> depois nós apontamos o cliente para a Groq.

###`python-dotenv`



Usada para gerenciar variáveis de ambiente e segredos de forma segura. Ela carrega chaves de API (como a OPENAI_API_KEY) a partir de um arquivo oculto chamado .env, evitando que você exponha suas chaves diretamente no código.

###`pydantic`

Biblioteca de validação de dados e estruturação de objetos baseada em tipos do Python. É amplamente utilizada em projetos de IA para garantir que as respostas fornecidas pelos modelos sigam um formato exato (como um JSON bem definido) através de Structured Outputs. (explicar Structured Outputs)

###`tenacity`

Biblioteca usada para implementar mecânicas de tentativa e erro (retry). Muito útil para chamadas de API, pois permite reconectar ou tentar novamente caso haja falhas de rede, taxas limite atingidas (rate limits) ou indisponibilidade temporária do serviço.

###`pdfplumber`

Ferramenta para extrair texto, tabelas e dados visuais de arquivos PDF. Muito comum em projetos que precisam ler documentos PDF para alimentar modelos de linguagem ou pipelines de RAG (Retrieval-Augmented Generation).

###`scikit-learn`

É ela que transforma os textos das leis em vetores numéricos e calcula o grau de relevância/relação entre a cláusula do contrato do usuário e os artigos da Lei do Inquilinato.

###`numpy`

Usado principalmente para operações com vetores e ordenação dos resultados.
É ela que ordena os resultados da busca por similaridade do maior score para o menor score, garantindo que o RAG recupere primeiro as leis mais parecidas com a consulta.

###`reportlab`

Usado para criar o PDF sintético de teste para que a biblioteca pdfplumper consiga ler e o LLM auditar.

##Configuração da API da Groq

In [82]:
# ============================================
# Configuração da API
# ============================================

import os
import json
import time
import types
import unicodedata

from openai import OpenAI

def obter_chave(nome: str) -> str:
    """
    Lê um segredo dos Secrets do Colab.
    Fora do Colab, utiliza variável de ambiente.
    """
    try:
        from google.colab import userdata
        return userdata.get(nome)

    except ImportError:
        valor = os.getenv(nome)

        if not valor:
            raise RuntimeError(
                f"Defina {nome} nos Secrets do Colab ou no ambiente."
            )

        return valor


LLM_BASE_URL = "https://api.groq.com/openai/v1"

LLM_API_KEY = obter_chave("GROQ_API_KEY")

LLM_MODEL = "openai/gpt-oss-120b"

PRECOS = {
    "openai/gpt-oss-20b": {
        "entrada": 0.075,
        "saida": 0.30
    },
    "openai/gpt-oss-120b": {
        "entrada": 0.150,
        "saida": 0.60
    }
}

TPM = 8_000


cliente = OpenAI(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)


print("✅ Chave carregada, termina em:", LLM_API_KEY[-4:])
print("🤖 Modelo:", LLM_MODEL)
print(f"📊 TPM do plano gratuito: {TPM:,}")

✅ Chave carregada, termina em: WtUg
🤖 Modelo: openai/gpt-oss-120b
📊 TPM do plano gratuito: 8,000


##Comunicação com o LLM

In [109]:
# ============================================
# Função de conversa com o LLM (abstração)
# ============================================

def conversar(
    mensagem_do_usuario: str,
    instrucao_de_sistema: str | None = None,
    temperatura: float = 0.0,
) -> tuple[str, dict]:
    """
    Envia uma mensagem ao modelo e devolve o texto da resposta
    junto com a estatística de uso de tokens.
    """

    mensagens = []

    if instrucao_de_sistema:
        mensagens.append({ "role": "system", "content": instrucao_de_sistema })

    mensagens.append({ "role": "user", "content": mensagem_do_usuario })

    resposta = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=mensagens,
        temperature=temperatura
    )

    return resposta.choices[0].message.content or ""

In [110]:
# ============================================
# Resiliência com Tenacity
# ============================================
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    reraise=True
)
def chamada_llm_com_retry(mensagens, response_format=None, temperatura=0.0):
    """
    Encapsula a chamada da API com política de resiliência e retentativas automáticas.
    """
    if response_format:
        return cliente.beta.chat.completions.parse(
            model=LLM_MODEL,
            messages=mensagens,
            response_format=response_format,
            temperature=temperatura
        )
    else:
        return cliente.chat.completions.create(
            model=LLM_MODEL,
            messages=mensagens,
            temperature=temperatura
        )

##Teste de conexão com a API

In [102]:
# ============================================
# Teste de sanidade da API
# ============================================

print("🔍 Testando conexão com a API...")

resposta_teste = conversar(
    mensagem_do_usuario=(
        "Responda em uma frase: "
        "O que é a Lei do Inquilinato?"
    ),
    instrucao_de_sistema=(
        "Seja conciso e responda em português do Brasil."
    )
)

print("✅ Conexão bem-sucedida!")
print(f"🤖 Resposta do Modelo: {resposta_teste}")

🔍 Testando conexão com a API...
✅ Conexão bem-sucedida!
🤖 Resposta do Modelo: ('A Lei do Inquilinato (Lei nº\u202f8.245/1991) regula as relações de locação de imóveis urbanos, estabelecendo direitos e deveres de locadores e locatários.', {'prompt_tokens': 104, 'completion_tokens': 100, 'total_tokens': 204})


##Modelo estruturado do contrato

In [122]:
# ============================================
# Modelo estruturado do contrato
# ============================================

from typing import Optional
from pydantic import BaseModel, Field


class DadosContrato(BaseModel):
    # Parte de Qualificação e Objeto
    locador_nome: str = Field(
        description="Nome completo do LOCADOR identificado no contrato"
    )
    locatario_nome: str = Field(
        description="Nome completo do LOCATÁRIO identificado no contrato"
    )
    endereco_imovel: str = Field(
        description="Endereço completo do imóvel objeto da locação"
    )

    # Valores e Encargos Financeiros
    valor_aluguel: float = Field(
        description="Valor mensal do aluguel em reais (ex: 2500.00)"
    )
    valor_condominio: float = Field(
        description="Valor da taxa ordinária de condomínio em reais (ex: 400.00)"
    )
    valor_iptu: float = Field(
        description="Valor do IPTU mensal em reais (ex: 100.00)"
    )
    clausula_fundo_reserva: str = Field(
        description=(
            "Resumo do texto sobre a responsabilidade do Fundo de Reserva "
            "e despesas extraordinárias de condomínio (ex: Parágrafo Único da Cláusula 3ª)"
        )
    )

    # Reajuste e Vigência
    indice_reajuste: str = Field(
        description="Índice de reajuste anual citado no contrato (ex: IGP-M, IPCA)"
    )
    prazo_meses: int = Field(
        description="Prazo total de duração da locação em meses (ex: 30)"
    )

    # Garantias e Cláusulas Críticas
    tipo_garantia: str = Field(
        description=(
            "Descrição detalhada das garantias exigidas no contrato "
            "(ex: Caução em dinheiro e/ou Fiador)"
        )
    )
    clausula_reformas: str = Field(
        description=(
            "Descrição da responsabilidade por benfeitorias, manutenção e reparos "
            "estruturais/vazamentos (ex: Parágrafo Único da Cláusula 6ª)"
        )
    )
    clausula_multa_rescisao: str = Field(
        description=(
            "Regras, percentuais e critérios de proporcionalidade da multa "
            "por rescisão antecipada (ex: Cláusula 7ª)"
        )
    )

##PDF sintético para teste

In [126]:
# ============================================
# Geração do contrato em PDF para teste
# ============================================

# Melhorar o contrato. Colocar um contrato mais completo

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


def gerar_contrato_sintetico_pdf(
    caminho_arquivo="contrato_exemplo.pdf"
):
    c = canvas.Canvas(
        caminho_arquivo,
        pagesize=letter
    )

    texto_contrato = [
        "INSTRUMENTO PARTICULAR DE CONTRATO DE LOCAÇÃO DE IMÓVEL RESIDENCIAL",
        "",
        "Pelo presente instrumento particular, de um lado:",
        "",
        "LOCADOR: CARLOS EDUARDO MENDES, brasileiro, casado, administrador, portador da Cédula de Identidade RG nº 12.345.678-SSP/ES e inscrito no CPF/MF sob o nº 123.456.789-00, residente e domiciliado na Rua das Palmeiras, nº 45, Vitória/ES;",
        "",
        "E, de outro lado:",
        "",
        "LOCATÁRIO: JOÃO PEDRO DA SILVA, brasileiro, solteiro, engenheiro de software, portador da Cédula de Identidade RG nº 98.765.432-SSP/ES e inscrito no CPF/MF sob o nº 987.654.321-11, residente e domiciliado na Cidade de Vila Velha/ES.",
        "",
        "As partes acima qualificadas têm, entre si, justo e contratado o presente Contrato de Locação Residencial, mediante as seguintes cláusulas e condições:",
        "",
        "CLÁUSULA 1ª – DO OBJETO E DESTINAÇÃO",
        "1.1 O objeto da presente locação é o imóvel residencial situado na Rua das Flores, nº 123, Bairro Centro, Cidade de Vitória, Estado do Espírito Santo, CEP 29000-000.",
        "1.2 O imóvel destina-se exclusivamente ao uso residencial do LOCATÁRIO e de sua família, sendo expressamente vedada a mudança de destinação, a sublocação, a cessão ou o empréstimo, no todo ou em parte, sem o prévio e formal consentimento do LOCADOR.",

        "CLÁUSULA 2ª – DO VALOR, VENCIMENTO E REAJUSTE",
        "2.1 O valor do aluguel mensal ajustado é de R$ 2.500,00 (dois mil e quinhentos reais), devendo ser pago impreterivelmente até o dia 05 (cinco) de cada mês subsequente ao vencido, mediante transferência bancária ou PIX na conta do LOCADOR.",
        "2.2 O valor do aluguel será reajustado a cada período de 12 (doze) meses, tomando-se como base a variação acumulada do IGP-M (Índice Geral de Preços - Mercado) divulgado pela Fundação Getúlio Vargas (FGV).",

        "CLÁUSULA 3ª – DAS TAXAS, CONDOMÍNIO E ENCARGOS",
        "3.1 Além do valor do aluguel, incumbe ao LOCATÁRIO o pagamento mensal dos encargos acessórios de locação, incluindo o Imposto Predial e Territorial Urbano (IPTU) no valor de R$ 100,00 mensais e a cota ordinária de Condomínio no valor fixado de R$ 400,00 mensais.",
        "3.2 PARÁGRAFO ÚNICO (DA COBRANÇA EXTRAORDINÁRIA): O LOCATÁRIO também arcará integralmente com as cobranças de Fundo de Reserva extraordinário, taxas de obras de expansão, pintura de fachada e melhorias estruturais aprovadas nas assembleias do condomínio durante todo o período da locação.",

        "CLÁUSULA 4ª – DO PRAZO DA LOCAÇÃO",
        "4.1 A presente locação é celebrada pelo prazo determinado de 30 (trinta) meses, iniciando-se na data de assinatura deste instrumento e encerrando-se na data correspondente após 30 meses, data em que o LOCATÁRIO obriga-se a restituir o imóvel totalmente livre e desocupado.",

        "CLÁUSULA 5ª – DA GARANTIA LOCATÍCIA",
        "5.1 Para garantir o fiel e exato cumprimento de todas as obrigações pecuniárias e legais assumidas neste contrato, exige-se do LOCATÁRIO a prestação de Caução em dinheiro no montante equivalente a 3 (três) aluguéis vigentes (R$ 7.500,00) E TAMBÉM a indicação de 01 (um) Fiador proprietário de imóvel quitado, sob pena de recusa da entrega das chaves.",

        "CLÁUSULA 6ª – DA CONSERVAÇÃO, MANUTENÇÃO E REFORMAS",
        "6.1 O LOCATÁRIO declara receber o imóvel no estado em que se encontra, comprometendo-se a mantê-lo limpo e conservado.",
        "6.2 PARÁGRAFO ÚNICO (DAS REFORMAS E ESTRUTURA): Fica pactuado que todas as benfeitorias necessárias e os reparos de caráter estrutural que se fizerem urgentes no imóvel — tais como vazamentos e infiltrações em tubulações centrais, vícios de construção ou rachaduras no telhado — serão de responsabilidade financeira exclusiva do LOCATÁRIO, sem direito a qualquer retenção ou indenização por parte do LOCADOR.",

        "CLÁUSULA 7ª – DA RESCISÃO ANTECIPADA E PENALIDADES",
        "7.1 Em caso de devolução do imóvel ou desocupação antecipada por iniciativa do LOCATÁRIO antes do término do prazo estipulado na Cláusula 4ª, este ficará obrigado ao pagamento de uma multa rescisória pré-fixada e irredutível correspondente a 50% (cinquenta por cento) do valor total dos aluguéis restantes até o final do contrato.",
        "7.2 A referida multa rescisória será aplicada em seu valor integral, sendo expressamente vedada qualquer redução proporcional em razão do tempo de contrato já cumprido pelo LOCATÁRIO.",

        "CLÁUSULA 8ª – DO FORO",
        "8.1 Para dirimir quaisquer questões oriundas do presente contrato, as partes elegem o Foro da Comarca de Vitória/ES, renunciando a qualquer outro, por mais privilegiado que seja.",

        "E, por estarem assim justos e contratados, assinam o presente instrumento em 02 (duas) vias de igual teor e forma.",

        "Vitória/ES, 10 de janeiro de 2026."
    ]

    y = 750  # Posição vertical inicial (topo da página)

    for linha in texto_contrato:
        # Se a página estiver acabando, cria uma nova página
        if y < 50:
            c.showPage()
            y = 750

        # Formatação simples para títulos vs corpo
        if "CLÁUSULA" in linha or "INSTRUMENTO" in linha or "CONTRATO" in linha:
            c.setFont("Helvetica-Bold", 10)
        else:
            c.setFont("Helvetica", 9)

        c.drawString(40, y, linha)
        y -= 15  # Espaçamento entre linhas

    c.save()

    print(
        f"📄 Contrato sintético criado com sucesso: "
        f"'{caminho_arquivo}'!"
    )


gerar_contrato_sintetico_pdf()

📄 Contrato sintético criado com sucesso: 'contrato_exemplo.pdf'!


##Leitura do PDF

In [124]:
# ============================================
# Leitura do PDF
# ============================================

import pdfplumber


def ler_pdf(caminho_pdf: str) -> str:
    """
    Extrai o texto bruto do PDF.
    """

    texto_completo = ""

    with pdfplumber.open(caminho_pdf) as pdf:

        for pagina in pdf.pages:

            texto_pagina = pagina.extract_text()

            if texto_pagina:
                texto_completo += texto_pagina + "\n"

    return texto_completo

##Teste de leitura do PDF

In [127]:
# ============================================
# Testando leitura do PDF
# ============================================

texto_bruto = ler_pdf("contrato_exemplo.pdf")

print("=" * 60)
print(f"📄 TEXTO EXTRAÍDO DO PDF ({len(texto_bruto)} caracteres):")
print("=" * 60)
print(texto_bruto)
print("=" * 60)

📄 TEXTO EXTRAÍDO DO PDF (4490 caracteres):
INSTRUMENTO PARTICULAR DE CONTRATO DE LOCAÇÃO DE IMÓVEL RESIDENCIAL
Pelo presente instrumento particular, de um lado:
LOCADOR: CARLOS EDUARDO MENDES, brasileiro, casado, administrador, portador da Cédula de Identidade RG nº 12.345.678-SSP/ES e inscrito no CPF/MF sob o nº 123.456.789-00, residente e domiciliado na Rua das Palmeiras, nº 45, Vitória/ES;
E, de outro lado:
LOCATÁRIO: JOÃO PEDRO DA SILVA, brasileiro, solteiro, engenheiro de software, portador da Cédula de Identidade RG nº 98.765.432-SSP/ES e inscrito no CPF/MF sob o nº 987.654.321-11, residente e domiciliado na Cidade de Vila Velha/ES.
As partes acima qualificadas têm, entre si, justo e contratado o presente Contrato de Locação Residencial, mediante as seguintes cláusulas e condições:
CLÁUSULA 1ª – DO OBJETO E DESTINAÇÃO
1.1 O objeto da presente locação é o imóvel residencial situado na Rua das Flores, nº 123, Bairro Centro, Cidade de Vitória, Estado do Espírito Santo, CEP 29000-000

##Extração estruturada usando o LLM

In [130]:
from httpx import Response

# ============================================
# Extração estruturada
# ============================================

def extrair_dados_estruturados(
    texto_contrato: str
) -> tuple[DadosContrato, dict]:
    """
    Utiliza o LLM com resiliência para extrair dados estruturados do contrato
    e contabilizar o gasto de tokens.
    """

    instrucao = (
        "Você é um especialista em análise e extração de dados de contratos imobiliários. "
        "Analise o texto completo fornecido e extraia com exatidão todos os campos do esquema. "
        "Preste especial atenção na qualificação das partes, no endereço, na identificação "
        "de cobranças de Fundo de Reserva/despesas extraordinárias e na cumulação de garantias."
    )

    mensagens = [
        {"role": "system", "content": instrucao},
        {"role": "user", "content": texto_contrato}
    ]

    resposta = chamada_llm_com_retry(
        mensagens=mensagens,
        response_format=DadosContrato,
        temperatura=0.0
    )

    tokens = {
        "prompt_tokens": resposta.usage.prompt_tokens,
        "completion_tokens": resposta.usage.completion_tokens,
        "total_tokens": resposta.usage.total_tokens
    }

    return resposta.choices[0].message.parsed, tokens

##Executar a extração

In [131]:
# ============================================
# Execução da extração
# ============================================

print("1. 📄 Lendo o PDF...")
texto_bruto = ler_pdf("contrato_exemplo.pdf")

print("2. 🤖 Extraindo dados estruturados via LLM...")
dados_extraidos, tokens_extracao = extrair_dados_estruturados(texto_bruto)

print("\n" + "=" * 60)
print("✅ EXTRAÇÃO ESTRUTURADA CONCLUÍDA")
print("=" * 60)

print("\n👥 PARTES E IMÓVEL:")
print(f"• Locador: {dados_extraidos.locador_nome}")
print(f"• Locatário: {dados_extraidos.locatario_nome}")
print(f"• Imóvel: {dados_extraidos.endereco_imovel}")

print("\n💰 VALORES E ENCARGOS:")
print(f"• Aluguel: R$ {dados_extraidos.valor_aluguel:.2f}")
print(f"• Condomínio (Ordinário): R$ {dados_extraidos.valor_condominio:.2f}")
print(f"• IPTU: R$ {dados_extraidos.valor_iptu:.2f}")
print(f"• Fundo de Reserva / Despesas Extraordinárias: {dados_extraidos.clausula_fundo_reserva}")

print("\n📅 REAJUSTE E PRAZO:")
print(f"• Reajuste: {dados_extraidos.indice_reajuste}")
print(f"• Prazo: {dados_extraidos.prazo_meses} meses")

print("\n⚖️ CLÁUSULAS REGULATÓRIAS E GARANTIAS:")
print(f"• Garantia Exigida: {dados_extraidos.tipo_garantia}")
print(f"• Reformas e Manutenção: {dados_extraidos.clausula_reformas}")
print(f"• Multa por Rescisão: {dados_extraidos.clausula_multa_rescisao}")

print("\n" + "-" * 60)
print(f"📊 Tokens nesta etapa: {tokens_extracao['total_tokens']:,} (Prompt: {tokens_extracao['prompt_tokens']:,} | Saída: {tokens_extracao['completion_tokens']:,})")
print("-" * 60)

1. 📄 Lendo o PDF...
2. 🤖 Extraindo dados estruturados via LLM...

✅ EXTRAÇÃO ESTRUTURADA CONCLUÍDA

👥 PARTES E IMÓVEL:
• Locador: CARLOS EDUARDO MENDES
• Locatário: JOÃO PEDRO DA SILVA
• Imóvel: Rua das Flores, nº 123, Bairro Centro, Cidade de Vitória, Estado do Espírito Santo, CEP 29000-000

💰 VALORES E ENCARGOS:
• Aluguel: R$ 2500.00
• Condomínio (Ordinário): R$ 400.00
• IPTU: R$ 100.00
• Fundo de Reserva / Despesas Extraordinárias: O LOCATÁRIO arcará integralmente com as cobranças de Fundo de Reserva extraordinário, bem como taxas de obras de expansão, pintura de fachada e melhorias estruturais aprovadas nas assembleias do condomínio durante todo o período da locação.

📅 REAJUSTE E PRAZO:
• Reajuste: IGP-M
• Prazo: 30 meses

⚖️ CLÁUSULAS REGULATÓRIAS E GARANTIAS:
• Garantia Exigida: Caução em dinheiro equivalente a 3 aluguéis (R$ 7.500,00) e fiador proprietário de imóvel quitado.
• Reformas e Manutenção: Todas as benfeitorias necessárias e os reparos de caráter estrutural urgentes, 

##Base jurídica

In [132]:
# ============================================
# Base de conhecimento jurídica
# ============================================

LEIS_E_JURISPRUDENCIAS = [
    # ----------------------------------------------------
    # LEI DO INQUILINATO (Lei nº 8.245/1991)
    # ----------------------------------------------------
    {
        "id": "art_37_dupla_garantia",
        "topico": "Garantia Locatícia / Vedação à Dupla Garantia",
        "texto": (
            "Artigo 37 da Lei nº 8.245/1991: Estabelece as modalidades válidas "
            "de garantia locatícia (caução, fiança, seguro de fiança locatícia e "
            "cessão fiduciária de quotas de fundo de investimento). O parágrafo "
            "único veda expressamente, sob pena de nulidade absoluta, a exigência "
            "de mais de uma modalidade de garantia em um mesmo contrato de locação."
        ),
    },
    {
        "id": "art_22_obrigacoes_locador",
        "topico": "Obras Estruturais, Vícios e Fundo de Reserva",
        "texto": (
            "Artigo 22 da Lei nº 8.245/1991: Define os deveres do locador, "
            "incluindo a responsabilidade exclusiva por vícios ou defeitos anteriores "
            "à locação, despesas extraordinárias de condomínio (obras de ampliação "
            "ou reforma estrutural, pintura de fachada, indenizações trabalhistas "
            "anteriores e constituição do Fundo de Reserva) e reparos essenciais à habitação."
        ),
    },
    {
        "id": "art_23_obrigacoes_locatario",
        "topico": "Despesas Ordinárias e Conservação do Imóvel",
        "texto": (
            "Artigo 23 da Lei nº 8.245/1991: Determina que o locatário deve "
            "pagar pontualmente o aluguel e encargos ordinários do condomínio "
            "(como manutenção preventiva, limpeza, consumo de água e luz das áreas "
            "comuns e salários dos empregados), além de zelar pelo imóvel e reparar "
            "danos decorrentes do uso inadequado."
        ),
    },
    {
        "id": "art_4_multa_proporcional",
        "topico": "Multa Rescisória e Proporcionalidade OBRIGATÓRIA",
        "texto": (
            "Artigo 4º da Lei nº 8.245/1991: Permite que o locatário devolva o imóvel "
            "antes do prazo estipulado mediante pagamento da multa contratual, a qual "
            "deve obrigatoriamente ser calculada de forma proporcional ao tempo restante "
            "para o término do contrato, sendo nula a cláusula que fixa multa de forma integral "
            "ou irredutível."
        ),
    },
    {
        "id": "art_17_18_reajuste_moeda",
        "topico": "Regras de Fixação e Reajuste do Aluguel",
        "texto": (
            "Artigos 17 e 18 da Lei nº 8.245/1991: Vedam a fixação do valor do "
            "aluguel em moeda estrangeira ou vinculado ao salário mínimo e à variação "
            "cambial. O reajuste deve ser anual e indexado a índice de preços oficial "
            "pactuado entre as partes (ex: IPCA, IGP-M)."
        ),
    },
    {
        "id": "art_35_36_benfeitorias",
        "topico": "Direito de Retenção e Indenização por Benfeitorias",
        "texto": (
            "Artigos 35 e 36 da Lei nº 8.245/1991: Salvo disposição contratual em contrário, "
            "as benfeitorias necessárias introduzidas pelo locatário, ainda que não autorizadas, "
            "bem como as úteis autorizadas, são indenizáveis e dão direito de retenção do imóvel."
        ),
    },
    # ----------------------------------------------------
    # CÓDIGO CIVIL (Lei nº 10.406/2002)
    # ----------------------------------------------------
    {
        "id": "cc_art_412_413_multa_penal",
        "topico": "Limites da Cláusula Penal e Redução Judicial",
        "texto": (
            "Artigos 412 e 413 do Código Civil: O valor da cominação imposta na "
            "cláusula penal não pode exceder o da obrigação principal. Ademais, o juiz "
            "deverá reduzir equitativamente a penalidade se a obrigação principal "
            "tiver sido cumprida em parte ou se o montante for manifestamente excessivo."
        ),
    },
    {
        "id": "cc_art_421_422_boa_fe",
        "topico": "Função Social do Contrato e Boa-Fé Objetiva",
        "texto": (
            "Artigos 421 e 422 do Código Civil: A liberdade contratual será exercida "
            "nos limites da função social do contrato. Os contratantes são obrigados a "
            "guardar, assim na conclusão do contrato, como em sua execução, os princípios "
            "de probidade e boa-fé."
        ),
    },
    {
        "id": "cc_art_566_568_locacao_geral",
        "topico": "Garantia de Uso Pacifico e Vícios Ocultos",
        "texto": (
            "Artigos 566 a 568 do Código Civil: O locador é obrigado a entregar ao "
            "locatário a coisa alugada em estado de servir ao uso a que se destina e a "
            "garantir-lhe, durante o tempo do contrato, o uso pacífico da coisa, "
            "respondendo por seus vícios ou defeitos ocultos."
        ),
    },
    # ----------------------------------------------------
    # CÓDIGO DE DEFESA DO CONSUMIDOR & APLICABILIDADE
    # ----------------------------------------------------
    {
        "id": "stj_sumula_aplicacao_cdc",
        "topico": "Inaplicabilidade do CDC na Relação Locatícia Direta",
        "texto": (
            "Jurisprudência Consolidada do STJ (Tema do CDC na Locação): O Código de Defesa "
            "do Consumidor (Lei 8.078/90) não se aplica aos contratos de locação residencial "
            "diretos regulados pela Lei 8.245/91, pois as relações locatícias possuem legislação "
            "própria. O CDC aplica-se, contudo, à relação entre o proprietário/inquilino e a "
            "imobiliária (prestação de serviços de intermediação)."
        ),
    },
    # ----------------------------------------------------
    # CONSTITUIÇÃO FEDERAL DE 1988
    # ----------------------------------------------------
    {
        "id": "cf_art_6_direito_moradia",
        "topico": "Direito Fundamental à Moradia",
        "texto": (
            "Artigo 6º da Constituição Federal de 1988: São direitos sociais a educação, "
            "a saúde, a alimentação, o trabalho, a moradia, o transporte, o lazer, a "
            "segurança, a previdência social, a proteção à maternidade e à infância."
        ),
    },
    {
        "id": "stf_tema_1098_fiador_penhora",
        "topico": "Penhorabilidade do Bem de Família do Fiador",
        "texto": (
            "Jurisprudência do STF (Tema 1098 / Súmula 549 do STJ): É constitucional a "
            "penhora de bem de família pertencente a fiador de contrato de locação, "
            "seja ela residencial ou comercial, nos termos do art. 3º, VII, da Lei 8.009/1990."
        ),
    },
]

##Criar o índice TF-IDF

###O que é TF-IDF?

**TF-IDF significa:**

Term Frequency — Inverse Document Frequency

**O que ele faz?**

Ele transforma textos em representação numérica.
A ideia intuitiva é que palavras importantes para um documento recebem maior peso, enquanto palavras muito comuns entre vários documentos recebem menor peso.

**Por exemplo:**

`caução - fiador - garantia` **
serão relevantes para o documento (presentes no Art. 37).

Enquanto palavras muito genéricas terão menos poder de diferenciação.

In [133]:
# ============================================
# Indexação da base jurídica
# ============================================

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


corpus_textos = [
    documento["texto"]
    for documento in LEIS_E_JURISPRUDENCIAS
]


vectorizer = TfidfVectorizer()


print(
    "🔍 Gerando matriz vetorial "
    "para os artigos da base jurídica..."
)


vetores_leis = vectorizer.fit_transform(
    corpus_textos
)


print(
    f"✅ Base legal indexada! "
    f"Total de {len(LEIS_E_JURISPRUDENCIAS)} "
    f"artigos convertidos em vetores."
)

🔍 Gerando matriz vetorial para os artigos da base jurídica...
✅ Base legal indexada! Total de 12 artigos convertidos em vetores.


##Função correta de Retrieval

In [139]:
# ============================================
# Retrieval do RAG
# ============================================

def buscar_fundamento_legal(
    query: str,
    top_k: int = 3,
    limite_similaridade: float = 0.05
) -> list[dict]:
    """
    Busca na base jurídica os artigos mais
    relevantes para a consulta utilizando
    TF-IDF + similaridade de cosseno.
    """

    # 1. Transforma a consulta em vetor
    vetor_query = vectorizer.transform(
        [query]
    )

    # 2. Calcula a similaridade entre
    #    consulta e todos os documentos
    similaridades = cosine_similarity(
        vetor_query,
        vetores_leis
    )[0]

    # 3. Ordena do maior para o menor score
    indices_top_k = np.argsort(
        similaridades
    )[::-1][:top_k]

    # 4. Monta os resultados
    resultados = []

    for idx in indices_top_k:

        score = float(similaridades[idx])

        if score < limite_similaridade:
            continue

        resultados.append({
            "artigo": LEIS_E_JURISPRUDENCIAS[idx],
            "score_similaridade": float(
                similaridades[idx]
            )
        })

        if len(resultados) >= top_k:
          break

    return resultados

##Testar o Retrieval

In [138]:
# ============================================
# Teste do Retrieval
# ============================================

print(
    "\n🔍 Testando o RAG:"
)

print(
    "Buscando leis aplicáveis sobre "
    "'exigir caução e fiador ao mesmo tempo'...\n"
)


busca_teste = buscar_fundamento_legal(
    "O contrato pede pagamento de caução e também fiador",
    top_k=2
)


for i, item in enumerate(
    busca_teste,
    start=1
):

    print(
        f"--- Resultado {i} "
        f"(Similaridade: "
        f"{item['score_similaridade']:.4f}) ---"
    )

    print(
        f"📌 Tópico: "
        f"{item['artigo']['topico']}"
    )

    print(
        f"📜 Texto Legal: "
        f"{item['artigo']['texto']}\n"
    )


🔍 Testando o RAG:
Buscando leis aplicáveis sobre 'exigir caução e fiador ao mesmo tempo'...

--- Resultado 1 (Similaridade: 0.2024) ---
📌 Tópico: Garantia Locatícia / Vedação à Dupla Garantia
📜 Texto Legal: Artigo 37 da Lei nº 8.245/1991: Estabelece as modalidades válidas de garantia locatícia (caução, fiança, seguro de fiança locatícia e cessão fiduciária de quotas de fundo de investimento). O parágrafo único veda expressamente, sob pena de nulidade absoluta, a exigência de mais de uma modalidade de garantia em um mesmo contrato de locação.

--- Resultado 2 (Similaridade: 0.1971) ---
📌 Tópico: Penhorabilidade do Bem de Família do Fiador
📜 Texto Legal: Jurisprudência do STF (Tema 1098 / Súmula 549 do STJ): É constitucional a penhora de bem de família pertencente a fiador de contrato de locação, seja ela residencial ou comercial, nos termos do art. 3º, VII, da Lei 8.009/1990.



#Resultados


In [141]:
# ============================================
# Resultados e analises
# ============================================

from pydantic import BaseModel, Field

# 1. Modelo Pydantic para estruturar a análise de cada cláusula
class AvaliacaoClausula(BaseModel):
    clausula_nome: str = Field(description="Nome da cláusula analisada")
    status: str = Field(description="Classificação: 'CONFORME', 'ATENÇÃO' ou 'ILEGAL'")
    pontuacao: float = Field(description="Pontuação de 0.0 (totalmente ilegal) a 1.0 (totalmente legal)")
    justificativa_simples: str = Field(description="Explicação em linguagem simples e direta para o leigo")
    fundamento_legal: str = Field(description="Citação do artigo de lei recuperado via RAG")

class RelatorioAuditoria(BaseModel):
    avaliacoes: list[AvaliacaoClausula]
    resumo_executivo: str = Field(description="Resumo geral dos riscos encontrados no contrato")

# 2. Função principal para auditoria automatizada
def auditar_contrato(dados: DadosContrato) -> dict:

    # Pontos críticos a serem auditados contra a Lei do Inquilinato
    itens_para_auditar = [
        {"nome": "Exigência de Garantias", "texto": dados.tipo_garantia, "busca": "dupla garantia caução fiador cumulação vedada"},
        {"nome": "Reformas e Obras Estruturais", "texto": dados.clausula_reformas, "busca": "obras reformas estruturais vazamento responsabilidade locador"},
        {"nome": "Despesas Condominiais e Fundo de Reserva", "texto": f"Condomínio R$ {dados.valor_condominio}", "busca": "fundo de reserva despesas extraordinarias locador"},
        {"nome": "Multa por Rescisão Antecipada", "texto": dados.clausula_multa_rescisao, "busca": "multa rescisao antecipada proporcionalidade tempo cumprido"},
        {"nome": "Índice de Reajuste", "texto": dados.indice_reajuste, "busca": "indice reajuste IGPM IPCA reajuste anual"}
    ]

    avaliacoes_resultados = []

    total_prompt_tokens = 0
    total_completion_tokens = 0

    for item in itens_para_auditar:
        # RAG Retrieval: Busca os artigos de lei mais próximos
        leis_relevantes = buscar_fundamento_legal(item["busca"], top_k=2)
        contexto_legal = "\n".join([f"- {l['artigo']['texto']}" for l in leis_relevantes])

        prompt_sistema = (
            "Você é um auditor jurídico especialista na Lei do Inquilinato (Lei 8.245/91).\n"
            "Analise a cláusula contratual em relação aos artigos da lei fornecidos.\n\n"
            "Regras de Pontuação:\n"
            "- Se for totalmente legal/conforme: status='CONFORME', pontuacao=1.0\n"
            "- Se for ambígua ou desfavorável sem ser explicitamente nula: status='ATENÇÃO', pontuacao=0.5\n"
            "- Se violar abertamente a lei (cláusula nula): status='ILEGAL', pontuacao=0.0\n\n"
            "Traduza a justificativa para linguagem simples e direta, sem 'juridiquês'."
        )

        prompt_usuario = (
            f"Cláusula/Item: {item['nome']}\n"
            f"Texto no Contrato: {item['texto']}\n\n"
            f"Base Legal Recuperada (RAG):\n{contexto_legal}"
        )

        mensagens = [
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario}
        ]

        resposta = chamada_llm_com_retry(
            mensagens=mensagens,
            response_format=AvaliacaoClausula,
            temperatura=0.0
        )

        total_prompt_tokens += resposta.usage.prompt_tokens
        total_completion_tokens += resposta.usage.completion_tokens

        avaliacoes_resultados.append(resposta.choices[0].message.parsed)

    # Cálculo do Índice de Confiabilidade do Contrato
    soma_pontos = sum([a.pontuacao for a in avaliacoes_resultados])
    total_itens = len(avaliacoes_resultados)
    porcentagem_confiabilidade = (soma_pontos / total_itens) * 100

    preco_modelo = PRECOS.get(LLM_MODEL, {"entrada": 0.150, "saida": 0.60})
    custo_entrada = (total_prompt_tokens / 1_000_000) * preco_modelo["entrada"]
    custo_saida = (total_completion_tokens / 1_000_000) * preco_modelo["saida"]
    custo_total_usd = custo_entrada + custo_saida

    return {
        "porcentagem_confiabilidade": porcentagem_confiabilidade,
        "avaliacoes": avaliacoes_resultados,
        "metricas_tokens": {
            "prompt_tokens": total_prompt_tokens,
            "completion_tokens": total_completion_tokens,
            "total_tokens": total_prompt_tokens + total_completion_tokens,
            "custo_estimado_usd": custo_total_usd
        }
    }

# 3. Execução e Exibição dos Resultados com Métricas
print("⏳ Executando auditoria legal e calculando índice de confiabilidade...\n")
resultado_auditoria = auditar_contrato(dados_extraidos)

score = resultado_auditoria["porcentagem_confiabilidade"]
metricas = resultado_auditoria["metricas_tokens"]

print("=" * 60)
print(f"📊 ÍNDICE DE CONFIABILIDADE DO CONTRATO: {score:.1f}%")
if score >= 80:
    print("🟢 Nível de Risco: BAIXO (Contrato equilibrado)")
elif score >= 50:
    print("🟡 Nível de Risco: MÉDIO (Requer ajustes e atenção)")
else:
    print("🔴 Nível de Risco: ALTO (Possui cláusulas abusivas ou nulas)")
print("=" * 60 + "\n")

print("📋 DETALHAMENTO DA AUDITORIA POR CLÁUSULA:\n")
for idx, item in enumerate(resultado_auditoria["avaliacoes"], 1):
    icone = "✅" if item.status == "CONFORME" else ("⚠️" if item.status == "ATENÇÃO" else "❌")
    print(f"{idx}. {icone} [{item.status}] - {item.clausula_nome}")
    print(f"   • Justificativa: {item.justificativa_simples}")
    print(f"   • Base Legal: {item.fundamento_legal}\n")

print("=" * 60)
print("📈 CONSUMO DE RECURSOS E TOKENS DA AUDITORIA:")
print(f"• Tokens de Entrada (Prompt): {metricas['prompt_tokens']:,}")
print(f"• Tokens de Saída (Completion): {metricas['completion_tokens']:,}")
print(f"• Total de Tokens Utilizados: {metricas['total_tokens']:,}")
print(f"• Custo Estimado da Operação: ${metricas['custo_estimado_usd']:.6f} USD")
print("=" * 60)

⏳ Executando auditoria legal e calculando índice de confiabilidade...

📊 ÍNDICE DE CONFIABILIDADE DO CONTRATO: 20.0%
🔴 Nível de Risco: ALTO (Possui cláusulas abusivas ou nulas)

📋 DETALHAMENTO DA AUDITORIA POR CLÁUSULA:

1. ❌ [ILEGAL] - Exigência de Garantias
   • Justificativa: A lei permite apenas uma forma de garantia (caução, fiança, seguro ou cessão fiduciária). O contrato pede caução e ainda um fiador, ou seja, duas garantias ao mesmo tempo, o que a lei proíbe, tornando a cláusula nula.
   • Base Legal: Art. 37 da Lei nº 8.245/1991, caput, e seu parágrafo único, que vedam a exigência de mais de uma modalidade de garantia em um mesmo contrato de locação.

2. ❌ [ILEGAL] - Reformas e Obras Estruturais
   • Justificativa: A lei manda que o locador pague reparos estruturais e conserte defeitos que já existiam; a cláusula coloca essa obrigação só no locatário, o que a lei não permite, então ela é nula.
   • Base Legal: Art. 22 da Lei nº 8.245/1991; arts. 566 a 568 do Código Civil

3. ⚠